# LeetCode 3646 — Next Special Palindrome Number bằng Python

- Link: https://leetcode.com/problems/next-special-palindrome-number/
- Độ khó: **Hard**
- Chủ đề: **Liệt kê có cấu trúc** (tập chữ số, hoán vị đa tập của nửa), palindrome, tìm nhị phân, "palindrome nhỏ nhất lớn hơn target"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def specialPalindrome(self, n: int) -> int:
```

## 1. Mục tiêu học

1. Phân tích điều kiện "chữ số `k` xuất hiện đúng `k` lần": số đặc biệt ↔ **tập chữ số** `S ⊆ {1..9}`, độ dài `= Σ S`.
2. Điều kiện palindrome: tối đa **một** chữ số có số lần lẻ ⇒ tối đa một chữ số **lẻ** trong `S`; nửa trái gồm `k // 2` bản của mỗi `k ∈ S`.
3. Đếm và **sinh toàn bộ** số đặc biệt (chỉ vài nghìn với độ dài ≤ 17), sắp xếp, `bisect` để trả lời.
4. Cách khác không cần sinh hết: với độ dài `L = len(n)`, mỗi tập `S` tìm palindrome nhỏ nhất `> n` bằng tham lam (như bài 3734); nếu không có, lấy số nhỏ nhất của độ dài `L + 1`.
5. Kiểm chứng bằng duyệt tuần tự với `n` nhỏ.

## 2. Đề bài & ví dụ

Số **đặc biệt**: là palindrome và mọi chữ số `k` xuất hiện **đúng `k` lần** (vì vậy không có chữ số 0). Tìm số đặc biệt nhỏ nhất **lớn hơn hẳn** `n`.

**Ví dụ 1:** `n = 2` → `22`.

**Ví dụ 2:** `n = 33` → `212`.

**Ràng buộc:** `0 ≤ n ≤ 10^15`.

In [1]:
from itertools import combinations, permutations
from bisect import bisect_right
from collections import Counter
import random, time

examples = [(2, 22), (33, 212)]
for n, exp in examples:
    print(n, "-> kỳ vọng", exp)

2 -> kỳ vọng 22
33 -> kỳ vọng 212


## 3. Kiến thức nền

### 3.1. Từ số đặc biệt về tập chữ số

Nếu chữ số `k` xuất hiện thì xuất hiện đúng `k` lần ⇒ đa tập chữ số được xác định bởi **tập** `S` các chữ số có mặt, và độ dài `L = Σ_{k∈S} k`. Chữ số 0 không thể xuất hiện (phải xuất hiện 0 lần).

In [2]:
def is_special(x):
    s = str(x)
    if s != s[::-1]: return False
    c = Counter(s)
    return all(int(d) == cnt for d, cnt in c.items())

print([x for x in range(1, 5000) if is_special(x)])

[1, 22, 212, 333, 4444]


### 3.2. Điều kiện palindrome

Palindrome ⇔ tối đa một chữ số có số lần xuất hiện lẻ. Chữ số `k` xuất hiện `k` lần ⇒ số lần lẻ ⇔ `k` lẻ. Vậy `S` chứa **tối đa một** số lẻ. Nửa trái = `k // 2` bản của mỗi `k ∈ S`; chữ số giữa = số lẻ duy nhất (nếu có).

In [3]:
valid_sets = [S for r in range(1, 10) for S in combinations(range(1, 10), r) if sum(1 for d in S if d % 2) <= 1]
print("số tập hợp lệ:", len(valid_sets), "| ví dụ:", valid_sets[:8])
print("độ dài lớn nhất:", max(sum(S) for S in valid_sets))

số tập hợp lệ: 95 | ví dụ: [(1,), (2,), (3,), (4,), (5,), (6,), (7,), (8,)]
độ dài lớn nhất: 29


### 3.3. Số lượng số đặc biệt nhỏ

Với mỗi `S`, số palindrome = số hoán vị đa tập của nửa trái = `(Σ k//2)! / Π (k//2)!`. Tổng qua các `S` có độ dài `≤ 17` chỉ vài nghìn ⇒ sinh hết được.

In [4]:
from math import factorial
def count_for(S):
    half = [d // 2 for d in S]; tot = factorial(sum(half))
    for h in half: tot //= factorial(h)
    return tot

print("tổng số đặc biệt độ dài ≤ 17:", sum(count_for(S) for S in valid_sets if sum(S) <= 17))

tổng số đặc biệt độ dài ≤ 17: 6986


### 3.4. Phân bố số đặc biệt theo độ dài

Độ dài `L` chỉ đạt được khi có tập `S` hợp lệ với `Σ S = L`. Bảng dưới cho thấy có những độ dài có rất ít số đặc biệt (ví dụ độ dài 2 chỉ có `22`), nên khoảng cách giữa hai số đặc biệt liên tiếp có thể rất lớn — lý do thử tuần tự `n+1, n+2, …` thất bại.

In [5]:
by_len = {}
for S in valid_sets:
    if sum(S) <= 17: by_len[sum(S)] = by_len.get(sum(S), 0) + count_for(S)
print({L: by_len.get(L, 0) for L in range(1, 18)})
print("độ dài không có số đặc biệt nào:", [L for L in range(1, 18) if by_len.get(L, 0) == 0])

{1: 1, 2: 1, 3: 2, 4: 1, 5: 4, 6: 4, 7: 11, 8: 5, 9: 32, 10: 15, 11: 95, 12: 75, 13: 335, 14: 140, 15: 1295, 16: 280, 17: 4690}
độ dài không có số đặc biệt nào: []


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đáp án có tối đa 17 chữ số (vì `n ≤ 10^15` có 16 chữ số và luôn tồn tại số đặc biệt 17 chữ số, ví dụ tập `{1, 2, 6, 8}` có tổng 17). Liệt kê mọi tập `S` hợp lệ với `Σ S ≤ 17`, sinh mọi hoán vị phân biệt của nửa trái, dựng palindrome, sắp xếp ⇒ `bisect_right` để lấy số đầu tiên `> n`.

Ví dụ 2: các số đặc biệt nhỏ: `1, 22, 212, 333, 1221, 2112, 4444, …` ⇒ số đầu tiên `> 33` là `212`. ✔

## 5. Cách 1 — Brute Force: thử `n+1, n+2, …`

In [6]:
def brute(n):
    x = n + 1
    while not is_special(x): x += 1
    return x

for n, exp in examples:
    print(brute(n), "kỳ vọng", exp)

22 kỳ vọng 22
212 kỳ vọng 212


**Nút thắt:** khoảng cách giữa các số đặc biệt có thể cực lớn (ví dụ sau `n ≈ 10^15`).

## 6. Ý tưởng thuật toán / bước đột phá

**Sinh toàn bộ** (Cách 2): `O(số đặc biệt · L)` một lần, mỗi truy vấn `O(log)`.

**Không sinh hết** (Cách 3): với `L = len(str(n))`, mỗi tập `S` có `Σ S = L` là một bài "palindrome nhỏ nhất lớn hơn target với đa tập cố định" — giải bằng tham lam tiền tố trùng dài nhất. Nếu không tập nào cho kết quả, đáp án là số đặc biệt nhỏ nhất có độ dài `> L`: duyệt `L' = L+1, L+2, …`, với mỗi tập lấy nửa trái **sắp tăng dần**.

## 7. Các cách tiếp theo

- **Cách 2:** sinh mọi số đặc biệt + `bisect` (lời giải nộp).
- **Cách 3:** tham lam theo từng tập chữ số (không sinh hết).
- **Cách 4:** sinh bằng **quay lui** trực tiếp theo thứ tự tăng của nửa trái, dừng ở số đầu tiên `> n` cho độ dài `L` (cắt nhánh khi tiền tố đã nhỏ hơn).

### 7.1. Cách 2 — Sinh toàn bộ (lời giải chính)

In [7]:
def all_special(max_len=17):
    out = []
    for r in range(1, 10):
        for S in combinations(range(1, 10), r):
            if sum(S) > max_len or sum(1 for d in S if d % 2) > 1: continue
            mid = "".join(str(d) for d in S if d % 2)
            half = "".join(str(d) * (d // 2) for d in S)
            for p in set(permutations(half)):
                h = "".join(p); out.append(int(h + mid + h[::-1]))
    return sorted(out)

SPECIAL = all_special()
def by_bisect(n):
    return SPECIAL[bisect_right(SPECIAL, n)]

print(len(SPECIAL), SPECIAL[:12])
for n, exp in examples:
    print(by_bisect(n), "kỳ vọng", exp)

6986 [1, 22, 212, 333, 4444, 23332, 32323, 44144, 55555, 244442, 424424, 442244]
22 kỳ vọng 22
212 kỳ vọng 212


### 7.2. Cách 3 — Tham lam theo từng tập chữ số

In [8]:
def smallest_pal_greater(S, target):
    # palindrome nhỏ nhất của tập S (độ dài = len(target)) và > target; None nếu không có
    L = len(target); m = L // 2
    mid = "".join(str(d) for d in S if d % 2)
    rem = Counter({str(d): d // 2 for d in S if d // 2})
    T = target[:m]
    build = lambda h: h + mid + h[::-1]
    p = 0
    while p < m and rem[T[p]] > 0: rem[T[p]] -= 1; p += 1
    if p == m:
        if build(T) > target: return build(T)
        if m == 0: return None
        p = m - 1; rem[T[p]] += 1
    for pos in range(min(p, m - 1), -1, -1):
        bigger = sorted(c for c in rem if rem[c] > 0 and c > T[pos])
        if bigger:
            c = bigger[0]; rem[c] -= 1
            return build(T[:pos] + c + "".join(ch * rem[ch] for ch in sorted(rem) if rem[ch] > 0))
        if pos > 0: rem[T[pos - 1]] += 1
    return None

def greedy_by_set(n):
    s = str(n); L = len(s)
    sets_by_len = {}
    for r in range(1, 10):
        for S in combinations(range(1, 10), r):
            if sum(1 for d in S if d % 2) <= 1: sets_by_len.setdefault(sum(S), []).append(S)
    best = None
    for S in sets_by_len.get(L, []):
        c = smallest_pal_greater(S, s)
        if c is not None and (best is None or int(c) < best): best = int(c)
    if best is not None: return best
    for L2 in range(L + 1, 46):
        cands = []
        for S in sets_by_len.get(L2, []):
            mid = "".join(str(d) for d in S if d % 2); h = "".join(str(d) * (d // 2) for d in S)
            cands.append(int(h + mid + h[::-1]))       # h đã tăng dần vì S tăng dần
        if cands: return min(cands)

for n, exp in examples:
    print(greedy_by_set(n), "kỳ vọng", exp)

22 kỳ vọng 22
212 kỳ vọng 212


### 7.3. Cách 4 — Quay lui theo thứ tự tăng

In [9]:
def backtrack(n):
    s = str(n); L0 = len(s)
    for L in range(L0, 46):
        best = None
        for r in range(1, 10):
            for S in combinations(range(1, 10), r):
                if sum(S) != L or sum(1 for d in S if d % 2) > 1: continue
                mid = "".join(str(d) for d in S if d % 2); cnt = {d: d // 2 for d in S if d // 2}; m = sum(cnt.values())
                target = s if L == L0 else ""
                def go(prefix, tight):
                    if len(prefix) == m:
                        pal = prefix + mid + prefix[::-1]
                        return pal if (not target or pal > target) else None
                    lo = target[len(prefix)] if (tight and target) else "0"
                    for d in sorted(cnt):
                        if cnt[d] == 0 or str(d) < lo: continue
                        cnt[d] -= 1
                        res = go(prefix + str(d), tight and str(d) == lo)
                        cnt[d] += 1
                        if res is not None: return res
                    return None
                res = go("", True)
                if res is not None and (best is None or int(res) < best): best = int(res)
        if best is not None: return best

for n, exp in examples:
    print(backtrack(n), "kỳ vọng", exp)

22 kỳ vọng 22
212 kỳ vọng 212


## 8. TRACE từng bước

In [10]:
def trace(n):
    s = str(n); L = len(s); print(f"n = {n}, độ dài {L}")
    for r in range(1, 10):
        for S in combinations(range(1, 10), r):
            if sum(S) == L and sum(1 for d in S if d % 2) <= 1:
                print(f"  tập {S}: palindrome nhỏ nhất > n cùng độ dài = {smallest_pal_greater(S, s)}")
    print("  đáp án:", by_bisect(n))

trace(33)
trace(4444)

n = 33, độ dài 2
  tập (2,): palindrome nhỏ nhất > n cùng độ dài = None
  đáp án: 212
n = 4444, độ dài 4
  tập (4,): palindrome nhỏ nhất > n cùng độ dài = None
  đáp án: 23332


## 9. Code nộp LeetCode

In [11]:
from itertools import combinations, permutations
from bisect import bisect_right

_SPECIAL = None

class Solution:
    def specialPalindrome(self, n: int) -> int:
        global _SPECIAL
        if _SPECIAL is None:
            out = []
            for r in range(1, 10):
                for S in combinations(range(1, 10), r):
                    if sum(S) > 17 or sum(d & 1 for d in S) > 1:
                        continue
                    mid = "".join(str(d) for d in S if d & 1)
                    half = "".join(str(d) * (d // 2) for d in S)
                    for p in set(permutations(half)):
                        h = "".join(p)
                        out.append(int(h + mid + h[::-1]))
            _SPECIAL = sorted(out)
        return _SPECIAL[bisect_right(_SPECIAL, n)]

sol = Solution()
for n, exp in examples:
    got = sol.specialPalindrome(n)
    print(n, "->", got, "✔" if got == exp else "✘")

2 -> 22 ✔
33 -> 212 ✔


## 10. Vì sao thuật toán đúng

1. **Đặc trưng:** số `x` đặc biệt ⇔ đa tập chữ số của `x` là `{k^k : k ∈ S}` với `S ⊆ {1..9}` và `x` là palindrome ⇔ `S` có tối đa một số lẻ và nửa trái của `x` là một hoán vị của `{k^{⌊k/2⌋}}`, chữ số giữa là số lẻ đó.
2. **Đủ:** sinh mọi `S` hợp lệ với `Σ S ≤ 17` và mọi hoán vị phân biệt ⇒ có mọi số đặc biệt tối đa 17 chữ số.
3. **Tồn tại đáp án ≤ 17 chữ số:** với `n ≤ 10^15 < 10^16`, mọi số 17 chữ số đều `> n`, và tồn tại số đặc biệt 17 chữ số (ví dụ `S = {1, 2, 6, 8}`).
4. `bisect_right(SPECIAL, n)` trả về vị trí phần tử đầu tiên `> n`.
5. **Cách 3:** palindrome cùng độ dài so sánh theo nửa trái (bài 3734) ⇒ tham lam đúng cho từng `S`; nếu không tập nào có số `> n` cùng độ dài, mọi số dài hơn đều `> n` ⇒ lấy nhỏ nhất ở độ dài nhỏ nhất có tập hợp lệ.

## 11. Phân tích Time / Space Complexity

Gọi `C` = số số đặc biệt `≤ 17` chữ số (vài nghìn):

| Cách | Time | Space |
|---|---|---|
| 1. Thử tuần tự | không bị chặn hữu ích | `O(1)` |
| 2. Sinh hết + `bisect` | `O(C · 17)` một lần, `O(log C)` mỗi truy vấn | `O(C)` |
| 3. Tham lam theo tập | `O(#tập · 17 · 10)` | `O(1)` |
| 4. Quay lui | tuỳ cắt nhánh, nhỏ | `O(17)` |

## 12. Bộ test

In [12]:
METHODS = [brute, by_bisect, greedy_by_set, backtrack, lambda n: Solution().specialPalindrome(n)]
for n, exp in examples:
    assert all(f(n) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [13]:
# 12.2 mọi n nhỏ so với thử tuần tự
for n in range(0, 3000):
    b = brute(n)
    assert by_bisect(n) == b == greedy_by_set(n) == backtrack(n), n
print("n = 0..2999: OK")

n = 0..2999: OK


In [14]:
# 12.3 n lớn ngẫu nhiên: ba cách nhanh phải khớp; đầu ra phải là số đặc biệt > n
rng = random.Random(3646)
for _ in range(300):
    n = rng.choice([rng.randint(0, 10**15), rng.choice(SPECIAL[:-1]), rng.choice(SPECIAL[:-1]) - 1])
    r = [by_bisect(n), greedy_by_set(n), backtrack(n), Solution().specialPalindrome(n)]
    assert len(set(r)) == 1 and r[0] > n and is_special(r[0]), (n, r)
print("300 n lớn: OK | n = 10^15 ->", Solution().specialPalindrome(10**15))

300 n lớn: OK | n = 10^15 -> 2666888888886662


In [15]:
# 12.4 thời gian: lần đầu (sinh) và các lần sau
import importlib
_SPECIAL = None
t0 = time.time(); Solution().specialPalindrome(10**15); t1 = time.time()
for _ in range(10**4): Solution().specialPalindrome(rng.randint(0, 10**15))
t2 = time.time()
print(f"lần đầu: {t1 - t0:.3f}s | 10^4 lần sau: {t2 - t1:.3f}s")

lần đầu: 0.030s | 10^4 lần sau: 0.008s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên điều kiện palindrome

Nếu chỉ kiểm tra "chữ số `k` xuất hiện đúng `k` lần" (số **cân bằng**), sau `33` ta được `122` — không đối xứng. Điều kiện palindrome là điều buộc chỉ được có **một** chữ số lẻ trong tập.

In [16]:
def wrong_no_palindrome(n):
    x = n + 1                                                           # SAI: chỉ kiểm tra "k xuất hiện k lần"
    while not all(int(d) == c for d, c in Counter(str(x)).items()): x += 1
    return x

print("sai:", wrong_no_palindrome(33), "| đúng:", brute(33))
assert wrong_no_palindrome(33) != brute(33)

sai: 122 | đúng: 212


### Lỗi 2 — Dùng `bisect_left` (trả về chính `n` khi `n` đặc biệt)

In [17]:
from bisect import bisect_left
print("sai:", SPECIAL[bisect_left(SPECIAL, 22)], "| đúng:", by_bisect(22))

sai: 22 | đúng: 212


### Lỗi 3 — Chỉ sinh tới 16 chữ số

In [18]:
SHORT = all_special(16)
n_big = max(x for x in SPECIAL if x < 10**16)
print("số đặc biệt lớn nhất ≤ 16 chữ số:", n_big)
print("với n = số đó, danh sách ≤ 16 chữ số không có đáp án:", bisect_right(SHORT, n_big) == len(SHORT),
      "| đáp án đúng:", by_bisect(n_big))

số đặc biệt lớn nhất ≤ 16 chữ số: 8888666226668888
với n = số đó, danh sách ≤ 16 chữ số không có đáp án: True | đáp án đúng: 23448888388884432


### Lỗi 4 — Coi chữ số 0 hợp lệ ("xuất hiện 0 lần")

Chữ số 0 nếu có mặt thì xuất hiện ≥ 1 lần ≠ 0 ⇒ không bao giờ hợp lệ; và số không thể bắt đầu bằng 0. Bỏ 0 khỏi tập chữ số.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Ràng buộc "chữ số `k` xuất hiện đúng `k` lần" ⇒ đếm được ⇒ **liệt kê** thay vì tìm kiếm.
- "Số nhỏ nhất > n thoả tính chất hiếm" ⇒ sinh hết các ứng viên (nếu ít) + `bisect_right`.
- Palindrome ⇒ làm việc trên nửa trái và một chữ số giữa.
- Kiểm tra độ dài tối đa cần sinh dựa trên ràng buộc của `n`.
- Khi danh sách ứng viên cố định cho mọi truy vấn, sinh **một lần** (biến toàn cục / thuộc tính lớp) để nhiều lần gọi `Solution()` không phải sinh lại.
- `set(permutations(...))` sinh trùng rồi khử trùng — chấp nhận được khi nửa trái ngắn (≤ 8 ký tự); nếu dài hơn nên dùng thuật toán sinh hoán vị đa tập (`next_permutation`).

## 15. Biến thể của bài

1. **Không cần palindrome** (LC 2048 *Next Greater Numerically Balanced Number*): sinh mọi hoán vị của đa tập.
2. **Số đặc biệt lớn nhất < n**: `bisect_left − 1`.
3. **Đếm số đặc biệt trong `[a, b]`**: hai lần `bisect`.
4. **Số đặc biệt thứ `k`**: chỉ số trong danh sách đã sắp.

In [19]:
# Biến thể 1: LC 2048 — số cân bằng (không cần palindrome) nhỏ nhất > n
def balanced_all(max_len=7):
    out = set()
    for r in range(1, 10):
        for S in combinations(range(1, 10), r):
            if sum(S) <= max_len:
                for p in set(permutations("".join(str(d) * d for d in S))): out.add(int("".join(p)))
    return sorted(out)

BAL = balanced_all()
def next_balanced(n): return BAL[bisect_right(BAL, n)]

def is_balanced(x): return all(int(d) == c for d, c in Counter(str(x)).items())
for n in range(0, 2000):
    x = n + 1
    while not is_balanced(x): x += 1
    assert next_balanced(n) == x
print("Biến thể 1 OK:", [next_balanced(n) for n in (1, 1000, 3000)])

Biến thể 1 OK: [22, 1333, 3133]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Thử tuần tự | Định nghĩa | rất chậm | Kiểm chứng `n` nhỏ |
| 2. Sinh hết + bisect | Liệt kê có cấu trúc | `O(C)` một lần | **Lời giải chuẩn** |
| 3. Tham lam theo tập | Như bài 3734 | `O(#tập · L)` | Khi không muốn sinh hết |
| 4. Quay lui | Cắt nhánh theo `n` | nhỏ | Mẫu tổng quát |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `specialPalindrome(0)`.

**Bài 2 (tính tay).** `specialPalindrome(212)`.

**Bài 3 (điền TODO).** Viết `count_special(a, b)` = số số đặc biệt trong `[a, b]`.

**Bài 4 (tự viết lại).** Viết `prev_special(n)` = số đặc biệt lớn nhất `< n` (trả `-1` nếu không có).

**Bài 5 (mini challenge).** Liệt kê các tập `S` cho số lượng palindrome đặc biệt **nhiều nhất** với độ dài `≤ 17`.

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_special(a, b):
    # TODO
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def richest_sets():
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `S = {1}` cho `1`. (2) `1 > 0`. (3) `1`.

**Bài 2.** (1) Độ dài 3: `S = {1, 2}` cho `212` (= `n`, không tính). (2) `S = {3}` cho `333`. (3) `333`.

**Bài 3.** (1) `bisect_right(b) − bisect_left(a)`. (2) Dùng `SPECIAL`. (3) Kiểm chứng với vòng lặp nhỏ.

**Bài 4.** (1) `i = bisect_left(SPECIAL, n) − 1`. (2) `i < 0` ⇒ `-1`. (3) Kiểm chứng.

**Bài 5.** (1) Đếm bằng công thức hoán vị đa tập. (2) Lấy max. (3) In các tập đạt max.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().specialPalindrome(0) == 1
assert Solution().specialPalindrome(212) == 333 == brute(212)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [23]:
# Bài 3, 4
from bisect import bisect_left
def count_special_sol(a, b):
    return bisect_right(SPECIAL, b) - bisect_left(SPECIAL, a)

def prev_special_sol(n):
    i = bisect_left(SPECIAL, n) - 1
    return SPECIAL[i] if i >= 0 else -1

for _ in range(300):
    a = rng.randint(0, 5000); b = rng.randint(a, 6000)
    assert count_special_sol(a, b) == sum(1 for x in range(a, b + 1) if is_special(x))
for n in range(0, 3000):
    x = n - 1
    while x >= 1 and not is_special(x): x -= 1
    assert prev_special_sol(n) == (x if x >= 1 else -1)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [24]:
# Bài 5
def richest_sets_sol():
    cands = [(count_for(S), S) for S in valid_sets if sum(S) <= 17]
    best = max(c for c, _ in cands)
    return best, [S for c, S in cands if c == best]

print("Bài 5:", richest_sets_sol())

Bài 5: (1680, [(2, 4, 5, 6)])


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
for S in tập con của {1..9}, sum(S) <= 17, tối đa một số lẻ:
    mid = số lẻ (nếu có); half = Σ str(k) * (k // 2)
    for mỗi hoán vị phân biệt h của half: thêm int(h + mid + h[::-1])
sắp xếp; return L[bisect_right(L, n)]
```

**Checklist:**

- [ ] Tối đa một chữ số lẻ trong tập?
- [ ] Sinh tới **17** chữ số (đủ cho `n ≤ 10^15`)?
- [ ] `bisect_right` (lớn hơn **hẳn**)?
- [ ] Loại chữ số 0?
- [ ] Đã so với thử tuần tự cho `n` nhỏ?